# Oppitunti 08 - Moni-agenttisuunnittelumalli


## Asennus


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Miksi moniagenttijärjestelmät?

Todellisen maailman tehtävät, kuten matkan suunnittelu, vaativat monenlaista asiantuntemusta — logistiikkaa, paikallistuntemusta, budjetointia ja muuta. Yksi agentti, joka yrittää hoitaa kaiken, muuttuu nopeasti hankalaksi hallita.

Moniagenttijärjestelmät ratkaisevat tämän **erikoistumisen** avulla: kukin agentti keskittyy yhteen asiantuntemuksen alueeseen ja tuottaa siten parempilaatuisia tuloksia kuin yleistietäjä. Ne parantavat myös **skaalautuvuutta** — voit lisätä uusia agentteja (esim. lentojen asiantuntija, ravintolakriitikko) ilman, että nykyistä työnkulkua tarvitsee kirjoittaa uudelleen. Agentit koostuvat yhdessä rakenteellisen putken kautta, välittäen kontekstia yhdeltä toiselle.


## Erikoistuneiden agenttien luominen


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Sekventiaalisen työprosessin rakentaminen

`WorkflowBuilder` antaa sinun yhdistää agentit suuntautuneeseen verkkoon. Tässä luomme yksinkertaisen kaksivaiheisen putken: **TravelPlanner** laatii matkasuunnitelman, ja sitten **TravelConcierge** tarkistaa ja parantaa sitä.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Lisätään lisää agenteja työnkulkuun

Yksi moniavustajamallin suurimmista eduista on sen helppo laajennettavuus. Alla lisäämme **BudgetReviewer**-agentin, joka tarkistaa suunnitelman matkustajan budjetin perusteella, merkitsee kohdat, jotka voivat ylittää kustannusrajan, ja ehdottaa rahansäästövaihtoehtoja. Työnkulku suorittaa nyt kolme agenttia peräkkäin:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Yhteenveto

Tässä oppitunnissa opit:

1. **Luomaan erikoistuneita agentteja** — jokaisella oma keskittynyt roolinsa (suunnittelu, palvelu, budjetin tarkistus).
2. **Kytkemään agentit peräkkäiseen työnkulkuun** käyttäen `WorkflowBuilder`-luokkaa ja `add_edge`-metodia.
3. **Virtauttamaan tulostetta** monen agentin putkistosta ja seuraamaan, mikä agentti puhuu.
4. **Laajentamaan työnkulkua** lisäämällä uusia agenteja ketjuun muokkaamatta olemassa olevia.

Monen agentin suunnittelumalli pitää jokaisen agentin yksinkertaisena, mutta tuottaa silti rikkaampia ja tarkemmin arvioituja tuloksia kuin yksikään agentti yksin pystyisi saavuttamaan.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
